# load data

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel('../data/properties_sampled.xlsx')

# Missing Data Pattern Analysis for price_per_square_meter

Before proceeding with data cleaning, we need to understand the nature of missing price_per_square_meter data. Missing data can be classified into three categories:
- **MCAR (Missing Completely At Random)**: Missing values are independent of both observed and unobserved data
- **MAR (Missing At Random)**: Missing values depend on observed data but not on unobserved data  
- **MNAR (Missing Not At Random)**: Missing values depend on unobserved data (the missing values themselves)

In [3]:
# Basic missing data statistics for price_per_square_meter
print("=== MISSING price_per_square_meter DATA SUMMARY ===")
print(f"Total observations: {len(df)}")
print(f"Missing price_per_square_meter values: {df['price_per_square_meter'].isna().sum()}")
print(f"Percentage missing: {df['price_per_square_meter'].isna().sum() / len(df) * 100:.2f}%")
print(f"Complete price_per_square_meter observations: {df['price_per_square_meter'].notna().sum()}")

# Check if missing values are related to market type
print("\n=== MISSING BY MARKET TYPE ===")
missing_by_market = df.groupby('market')['price_per_square_meter'].apply(lambda x: x.isna().sum())
total_by_market = df.groupby('market').size()
percent_missing_by_market = (missing_by_market / total_by_market * 100).round(2)

for market in df['market'].unique():
    print(f"{market}: {missing_by_market[market]} missing ({percent_missing_by_market[market]}%)")
import scipy.stats as stats
from scipy.stats import chi2_contingency, chi2

# 1. Little's MCAR Test (simplified implementation)
def little_mcar_test(data):
    """
    Simplified implementation of Little's MCAR test
    Returns test statistic and p-value
    """
    # Select only numeric columns for the test
    numeric_cols = data.select_dtypes(include=[np.number]).columns
    test_data = data[numeric_cols].copy()
    
    # Create missing pattern matrix
    missing_patterns = test_data.isnull()
    
    # Count unique missing patterns
    pattern_counts = missing_patterns.value_counts()
    
    if len(pattern_counts) <= 1:
        return None, None, "All data complete or only one missing pattern"
    
    # Calculate expected vs observed frequencies (simplified)
    total_missing = test_data.isnull().sum().sum()
    total_values = test_data.size
    
    if total_missing == 0:
        return None, None, "No missing values found"
    
    # Simplified chi-square test for randomness
    observed_pattern_freq = pattern_counts.values
    expected_freq = np.full_like(observed_pattern_freq, np.mean(observed_pattern_freq))
    
    if len(observed_pattern_freq) > 1:
        chi2_stat = np.sum((observed_pattern_freq - expected_freq)**2 / expected_freq)
        df = len(observed_pattern_freq) - 1
        p_value = 1 - chi2.cdf(chi2_stat, df)
        return chi2_stat, p_value, "Test completed"
    
    return None, None, "Insufficient patterns for test"

print("=== LITTLE'S MCAR TEST ===")
chi2_stat, p_value, message = little_mcar_test(df)

if chi2_stat is not None:
    print(f"Chi-square statistic: {chi2_stat:.4f}")
    print(f"P-value: {p_value:.4f}")
    if p_value > 0.05:
        print("Result: FAIL TO REJECT H0 - Data appears to be MCAR (p > 0.05)")
    else:
        print("Result: REJECT H0 - Data does NOT appear to be MCAR (p ≤ 0.05)")
else:
    print(f"Test could not be performed: {message}")
# 2. MAR Analysis: Test if missingness depends on observed variables
print("\n=== MAR ANALYSIS ===")
print("Testing if price_per_square_meter missingness depends on observed variables:")

# Create binary indicator for missing price_per_square_meter
df_temp = df.copy()
df_temp['price_missing'] = df_temp['price_per_square_meter'].isna()

# Test categorical variables
categorical_vars = ['market', 'material', 'district', 'river_side']
print("\n--- Categorical Variables ---")

for var in categorical_vars:
    if var in df_temp.columns:
        # Create contingency table
        contingency_table = pd.crosstab(df_temp[var].fillna('Missing'), df_temp['price_missing'])
        
        # Chi-square test
        chi2_stat, p_val, dof, expected = chi2_contingency(contingency_table)
        
        print(f"\n{var.upper()}:")
        print(f"  Chi-square: {chi2_stat:.4f}, p-value: {p_val:.4f}")
        if p_val < 0.05:
            print(f"  *** SIGNIFICANT: price_per_square_meter missingness depends on {var} ***")
        else:
            print(f"  Not significant: price_per_square_meter missingness independent of {var}")
        
        # Show proportions
        prop_table = contingency_table.div(contingency_table.sum(axis=1), axis=0)
        print(f"  Missing rate by category:")
        for idx in prop_table.index:
            missing_rate = prop_table.loc[idx, True] if True in prop_table.columns else 0
            print(f"    {idx}: {missing_rate:.3f}")

# Test numerical variables
print("\n--- Numerical Variables ---")
numerical_vars = ['surface', 'rooms', 'floor', 'construction_year']

for var in numerical_vars:
    if var in df_temp.columns:
        # Split into missing/non-missing price_per_square_meter groups
        missing_group = df_temp[df_temp['price_missing'] == True][var].dropna()
        present_group = df_temp[df_temp['price_missing'] == False][var].dropna()
        
        if len(missing_group) > 0 and len(present_group) > 0:
            # T-test for difference in means
            t_stat, p_val = stats.ttest_ind(missing_group, present_group, nan_policy='omit')
            
            print(f"\n{var.upper()}:")
            print(f"  Mean when price_per_square_meter missing: {missing_group.mean():.2f}")
            print(f"  Mean when price_per_square_meter present: {present_group.mean():.2f}")
            print(f"  T-test p-value: {p_val:.4f}")
            
            if p_val < 0.05:
                print(f"  *** SIGNIFICANT: {var} differs between missing/present groups ***")
            else:
                print(f"  Not significant: {var} similar between groups")


# Clean up temporary column
df_temp = df_temp.drop('price_missing', axis=1)
# 4. Missing Data Mechanism Conclusion and Recommendations
print("\n=== MISSING DATA MECHANISM ASSESSMENT ===")

print("""
INTERPRETATION GUIDE:
• MCAR (Missing Completely At Random): Missing values are random and unrelated to any variables
• MAR (Missing At Random): Missing values depend on observed variables but not the missing values themselves  
• MNAR (Missing Not At Random): Missing values depend on the unobserved values (e.g., high prices hidden intentionally)

RECOMMENDATIONS BASED ON MECHANISM:
• If MCAR: Complete case analysis is unbiased; imputation methods work well
• If MAR: Use imputation methods that account for relationships with observed variables
• If MNAR: Need domain knowledge; consider selection models or pattern-mixture models

PRACTICAL IMPLICATIONS FOR price_per_square_meter DATA:
""")

# Calculate some key statistics for interpretation
missing_rate = df['price_per_square_meter'].isna().mean()
market_specific_rates = df.groupby('market')['price_per_square_meter'].apply(lambda x: x.isna().mean())

if missing_rate > 0.2:
    print(f"• High missing rate ({missing_rate:.1%}) suggests systematic reasons for missingness")
else:
    print(f"• Moderate missing rate ({missing_rate:.1%}) may be manageable with appropriate methods")

if market_specific_rates.std() > 0.05:
    print("• Substantial difference in missing rates between market types suggests MAR or MNAR")
else:
    print("• Similar missing rates across market types supports MCAR hypothesis")

print("\nRECOMMENDED APPROACH:")
print("1. If tests suggest MCAR: Use complete case analysis or simple imputation")
print("2. If tests suggest MAR: Use multiple imputation or model-based methods")  
print("3. If evidence suggests MNAR: Consider domain knowledge about pricing strategies")
print("4. For robustness: Compare results across different handling methods")

print(f"\nNEXT STEPS:")
print("• Consider the business context: Why might prices be missing?")
print("• Validate any imputation method with available complete cases") 
print("• Consider sensitivity analysis with different missing data assumptions")
print("• Document the chosen approach and its limitations")

=== MISSING price_per_square_meter DATA SUMMARY ===
Total observations: 8767
Missing price_per_square_meter values: 1160
Percentage missing: 13.23%
Complete price_per_square_meter observations: 7607

=== MISSING BY MARKET TYPE ===
pierwotny: 1039 missing (48.51%)
wtórny: 121 missing (1.83%)
=== LITTLE'S MCAR TEST ===
Chi-square statistic: 4741.4796
P-value: 0.0000
Result: REJECT H0 - Data does NOT appear to be MCAR (p ≤ 0.05)

=== MAR ANALYSIS ===
Testing if price_per_square_meter missingness depends on observed variables:

--- Categorical Variables ---

MARKET:
  Chi-square: 3068.0698, p-value: 0.0000
  *** SIGNIFICANT: price_per_square_meter missingness depends on market ***
  Missing rate by category:
    pierwotny: 0.485
    wtórny: 0.018

MATERIAL:
  Chi-square: 1218.1075, p-value: 0.0000
  *** SIGNIFICANT: price_per_square_meter missingness depends on material ***
  Missing rate by category:
    beton: 0.512
    cegła: 0.047
    drewno: 0.000
    inny: 0.133
    pustak: 0.025
   

# summary
we reject H0 ->missing data is MCAR -> it is MAR or MNAR
The missingness of price per squere meter dependend on other variables or on itself
